# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery
- LangSmith evaluation & monitoring

## 1. Installation & Setup

In [41]:
!pip install -q langchain langchain-openai langchain-anthropic langsmith pydantic gradio

## 2. Configuration

**Edit the values below before running the notebook.**

from google.colab import userdata
userdata.get('secretName')

In [42]:
# ============================================================
# CONFIGURATION 
# ============================================================
from meal_planner_config import (
    OPENAI_API_KEY, LANGSMITH_API_KEY, OPENAI_MODEL, TEMPERATURE,
    LANGSMITH_PROJECT, EMAIL_SENDER, EMAIL_PASSWORD,
    INGREDIENTS_PER_RECIPE, EXCLUDED_BASICS, DAYS, validate_config
)

validate_config(require_email=False, require_langsmith=False)

In [43]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT

## 3. Pydantic Models for Validation

In [44]:
from models import DietaryPreference, Ingredient, Recipe, ShoppingItem, MealPlan


## 4. LangChain Setup & Prompts

In [45]:
from prompts import MEAL_PLAN_PROMPT, SHOPPING_LIST_PROMPT, get_llm

## 5. Meal Generation Logic

In [46]:
from meal_plan_generator import MealPlanGenerator
generator = MealPlanGenerator()

## 6. Email Functionality

In [47]:
from email_utils import format_meal_plan_email, send_meal_plan_email

## 7. LangSmith Evaluations

Custom evaluators for:
- Shopping list completeness
- Theme uniqueness and 5-ingredient rule
- No hallucinated ingredients

In [48]:
from langsmith import Client
from evaluators import shopping_list_completeness, theme_uniqueness, dietary_compliance

ls_client = Client()

/opt/anaconda3/envs/bootcamp-env/lib/python3.11/site-packages/langsmith/client.py:656: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [49]:
# ============================================================
# TEST DATASET & LANGSMITH EVALUATION
# ============================================================
from dataset_evaluation import run_basic_validation, run_langsmith_evaluation

# LangSmith evaluation requires LANGSMITH_API_KEY — skip gracefully if not set
if LANGSMITH_API_KEY:
    evaluation_results = run_langsmith_evaluation(ls_client, generator)
else:
    print("Skipping LangSmith evaluation: LANGSMITH_API_KEY not set.")

Skipping LangSmith evaluation: LANGSMITH_API_KEY not set.


## 8. Gradio Interface

In [50]:
from app import build_app, main
from meal_plan_generator import MealPlanGenerator

# For interactive notebook use (not the CLI entrypoint):
generator = MealPlanGenerator()
app = build_app(generator)

## 9. Launch Application

In [51]:
# Launch the Gradio app
# In Colab, this will create a public URL
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://e2c55745534af05bfc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Run LLM-as-Judge Evaluation

Run this cell to execute evaluation with LLM-as-judge evaluators in LangSmith.
The evaluators check:
- **Shopping list completeness**: All recipe ingredients in shopping list
- **Theme uniqueness**: 7 unique themes, 5 ingredients each
- **Dietary compliance**: Real ingredients, no dietary violations

In [52]:
# Run LLM-as-judge evaluation in LangSmith
# This will create a dataset and run all 3 evaluators on each test case
evaluation_results = run_langsmith_evaluation()

TypeError: run_langsmith_evaluation() missing 2 required positional arguments: 'ls_client' and 'generator'

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019f7f7d-509b-7622-8d3d-f272543ee0eb,id=019f7f7d-509b-7622-8d3d-f272543ee0eb; trace=019f7f7d-509b-7622-8d3d-f272543ee0eb,id=019f7f7d-509c-7b80-b639-745c423995ae; trace=019f7f7d-509b-7622-8d3d-f272543ee0eb,id=019f7f7d-509d-72f0-b78b-6b97ccf8819c; trace=019f7f7d-509b-7622-8d3d-f272543ee0eb,id=019f7f7d-509d-72f0-b78b-6b97ccf8819c; trace=019f7f7d-509b-7622-8d3d-f272543ee0eb,id=019f7f7d-509f-7222-bd0a-2c19ff9b4155
Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019f7f7d-